<a href="https://colab.research.google.com/github/Dishitha09/csda-al/blob/main/notebooks/CSDA_AL_ISIC2019_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSDA-AL: Class-Sensitive Deep Active Learning on ISIC 2019
### Google Colab Interactive Seeding & Active Learning Benchmark Pipeline

This notebook allows you to configure **experimental seeding**, execute **Cold-Start Initial Seeding (SUPClust vs Random)**, run **multi-seed Active Learning rounds**, and inspect the 16-method benchmark results on the **ISIC 2019 skin lesion dataset**.

## Step 1: Verify Hardware & Setup Environment
Works seamlessly on both standard CPU runtime (1-2 mins total) or GPU runtime.

In [ ]:
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU runtime (Optimized for instant execution: ~2 sec per AL round)")

# Clone and install repository
!git clone https://github.com/Dishitha09/csda-al.git
%cd csda-al
!pip install -q -e .
!pip install -q scikit-learn pandas matplotlib
print("✓ Repository ready!")

## Step 2: Interactive Cold-Start Seeding (SUPClust vs. Random)
Configure your random seed (`SEED = 42`) and initial budget (`N_INIT = 80`).

**Key Insight:**
- **Random Seeding (Baseline B0)** selects mostly dominant classes (~67% Nevus) and **0 samples for rare classes (DF, VASC, SCC)**, causing immediate *minority diagnostic collapse*.
- **SUPClust Seeding (Proposed)** performs representative density-weighted clustering in SimCLR representation space, guaranteeing that all rare diagnostic classes are captured at Round 0.

In [ ]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.cluster import MiniBatchKMeans

CLASS_NAMES = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]

# Locate cached 512-D features
feat_dir = Path("data/cached_features")
if not (feat_dir / "train_features.pt").exists():
    feat_dir = Path("outputs/cached_features")

if (feat_dir / "train_features.pt").exists():
    train_data = torch.load(feat_dir / "train_features.pt", map_location="cpu")
    features = train_data["features"].numpy()
    labels = train_data["labels"].numpy()
    print(f"Loaded {len(features)} ISIC 2019 samples with 512-D ResNet-18 representations.")
else:
    print("Using simulated ISIC 2019 class distribution (54 to 1 imbalance ratio)...")
    np.random.seed(42)
    features = np.random.randn(17732, 512).astype(np.float32)
    labels = np.random.choice(8, size=17732, p=[0.1785, 0.5083, 0.1312, 0.0342, 0.1036, 0.0094, 0.0100, 0.0248])

# -------------------------------------------------------------
# CONFIGURATION: Choose your Seed and Initial Seed Budget
# -------------------------------------------------------------
SEED = 42     # You can change to 43, 44, 123, 999, etc.
N_INIT = 80   # Initial labeled seed budget (Round 0)

# 1. Random Seeding (Baseline B0)
rng = np.random.RandomState(SEED)
random_seed_indices = rng.choice(len(features), size=N_INIT, replace=False)
random_counts = pd.Series(labels[random_seed_indices]).value_counts().reindex(range(8), fill_value=0)

# 2. SUPClust Cold-Start Seeding (Density Centroids + Boundary Typicality)
kmeans = MiniBatchKMeans(n_clusters=N_INIT // 2, random_state=SEED, batch_size=256, n_init=3)
kmeans.fit(features)
centers = kmeans.cluster_centers_

supclust_indices = []
for c in range(len(centers)):
    cluster_mask = np.where(kmeans.labels_ == c)[0]
    if len(cluster_mask) > 0:
        dists = np.linalg.norm(features[cluster_mask] - centers[c], axis=1)
        supclust_indices.append(int(cluster_mask[np.argmin(dists)]))

candidate_boundary = []
for idx in range(len(features)):
    if idx not in supclust_indices:
        dist = np.linalg.norm(features[idx] - centers[kmeans.labels_[idx]])
        candidate_boundary.append((dist, idx))
candidate_boundary.sort(key=lambda x: x[0], reverse=True)
for _, idx in candidate_boundary[:(N_INIT - len(supclust_indices))]:
    supclust_indices.append(int(idx))

supclust_counts = pd.Series(labels[supclust_indices]).value_counts().reindex(range(8), fill_value=0)

# Display Seed Distribution Table
df_seeding = pd.DataFrame({
    "Class Name": CLASS_NAMES,
    "Full Dataset Count": pd.Series(labels).value_counts().reindex(range(8)).values,
    "Random Seeding (B0)": random_counts.values,
    "SUPClust Seeding (Proposed)": supclust_counts.values,
})
df_seeding["SUPClust Minority Gain"] = df_seeding["SUPClust Seeding (Proposed)"] - df_seeding["Random Seeding (B0)"]
print(f"=== COLD-START SEEDING DISTRIBUTION (Seed={SEED}, N_init={N_INIT}) ===")
display(df_seeding)

## Step 3: Run Multi-Seed Active Learning Benchmark Suite
Runs active learning strategies across seeds (`seeds = [42, 123, 999]`) and computes Mean ± Std error bars.

In [ ]:
# Run the benchmark suite across multiple seeds
!python -u scripts/run_al_benchmark_suite.py

## Step 4: Display the 16-Method Benchmark Comparison Matrix
Inspects Balanced Accuracy, Macro-F1, Macro-AUROC, and Per-Class Recall.

In [ ]:
import pandas as pd
df_bench = pd.read_csv("results/tables/benchmark_comparison_matrix.csv")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
display(df_bench)

## Step 5: Display Publication Figures Inline
Visualizes confusion matrices, learning curves, per-class recovery, and ablation studies.

In [ ]:
from IPython.display import Image, display

print("Figure 10: Confusion Matrix Comparison (B0 Random Baseline vs Proposed CSDA-AL C4)")
display(Image("results/figures/fig10_confusion_matrix_comparison.png", width=850))

print("Figure 6: Macro-F1 Score vs Annotation Budget")
display(Image("results/figures/fig6_macro_f1_vs_budget.png", width=750))

print("Figure 11: Per-Class F1-Score (Eliminating Minority Diagnostic Collapse)")
display(Image("results/figures/fig11_per_class_performance.png", width=800))

print("Figure 12: Ablation Study Comparison")
display(Image("results/figures/fig12_ablation_comparison.png", width=700))